# Metabolic Forest — product door (`xenosite.forest`)

This notebook walks through the **Rust chematic** public API (PyO3 handles, not dicts).
Examples use realistic drug-like substrates from the crate gold tests.

Code cells **print** a few scalars via attribute access, then end with a handle so
IPython shows the rich `_repr_html_` / `__str__` (ForestMol tag tracing, plans,
network summaries — not hop dumps).

**Prerequisites:** extension built (`uv sync` or `maturin develop --features python,extension-module`).
See [docs/forest/RUST.md](../docs/forest/RUST.md).

In [1]:
from xenosite.forest import (
    ForestMol,
    MetabolicNetwork,
    available,
    default_ruleset,
    find_path,
    find_path_partial,
    normalize_tautomer,
    phase_one,
    product_graph_bfs,
    product_graph_ruleset,
    random_path,
)

assert available(), "Build xenosite.forest._rust (maturin develop)"
print("extension OK — cells end with a handle so IPython shows _repr_html_")
ForestMol("CCO")

extension OK — cells end with a handle so IPython shows _repr_html_


ForestMol("C(C)O")

## 1. `find_path` — acetaminophen toward NAPQI

Acetaminophen (APAP) bioactivation to **NAPQI** is a classic Phase I tox pathway
(hydroxylation / quinone-imine chemistry). We search with **`default_ruleset()`**
(includes tautomerization) or **`phase_one()`** for a Phase I–only catalog.

In [2]:
APAP = "CC(=O)Nc1ccc(O)cc1"
NAPQI = "CC(=O)N=C1C=CC(=O)C=C1"

hits, counters, net = find_path(
    APAP,
    NAPQI,
    max_paths=2,
    max_nodes=80,
    use_atom_diff=False,
)
print(
    f"hits={len(hits)} billed={counters.billed} nodes={net.n_nodes()} edges={net.n_edges()}"
)
hits[0]  # PathOutcome pathway (SOM on each reactant)

hits=2 billed=755 nodes=19 edges=20


PathOutcome("C=1C(C=CC(C=1)=O)=NC(C)=O")

**Handles:** `hit.mol` is a tagged `ForestMol`; `hit.plan` is a `StepPlan` (elementary deps).
Last expression above is the `PathOutcome` display. Export-only JSON: `hit.to_dict()`.
Next cell peels the plan for a closer look.

In [3]:
h = hits[0]
print("plan length:", len(h.plan), "~lin:", h.plan.n_linearizations())
print("stamp_end / survivors / born / untagged:", h.mol.trace_counts())
h.plan  # StepPlan (linearizations when small)

plan length: 1 ~lin: 1
stamp_end / survivors / born / untagged: (12, 11, 0, 0)


StepPlan(1 steps, ~1 lin)

## 2. `find_path` — terbinafine N-dealkylation

Terbinafine loses the **N-methyl–naphthyl** arm to give an aldehyde metabolite (TBA).
Gold test uses `phase_one()` and a stereo-specific target spelling from chematic.

In [4]:
TERBINAFINE = "CN(C/C=C/C#CC(C)(C)C)Cc1cccc2ccccc12"
TBF_ALDEHYDE = r"C(#C/C=C/C=O)C(C)(C)C"

hits, counters, net = find_path(
    TERBINAFINE,
    TBF_ALDEHYDE,
    max_paths=1,
    max_nodes=40,
    use_atom_diff=False,
)
assert hits, "expected dealkylation path"
print("first step:", hits[0].to_dict()["steps"][0], "graph:", net.n_nodes(), "nodes")
hits[0]

first step: {'rule': 'Dealkylation', 'site': ['6']} graph: 40 nodes


PathOutcome("C(#C/C=C/C=O)C(C)(C)C")

## 3. `find_path_partial` — exact hits plus closest misses

When the target is unreachable within `max_nodes`, partial outcomes carry a **residual**
(atom-diff style gap) for the closest products on the graph.

In [5]:
exact, partials, counters, net = find_path_partial(
    APAP,
    NAPQI,
    max_paths=1,
    max_nodes=80,
    use_atom_diff=False,
)
print(
    f"exact={len(exact)} partial={len(partials)} billed={counters.billed} graph={net.n_nodes()}n/{net.n_edges()}e"
)
# Prefer a partial when present; else an exact hit.
partials[0] if partials else exact[0]

exact=1 partial=0 billed=265 graph=2n/1e


PathOutcome("C=1C(C=CC(C=1)=O)=NC(C)=O")

## 4. One-step **`metabolize`** on APAP (Phase I catalog)

`RuleSet.metabolize` returns **`Emission`** handles and accepts the public
`ForestMol` wrapper or the extension pyclass.

In [6]:
mol = ForestMol(APAP)
emissions = phase_one().metabolize(mol)
print(f"{len(emissions)} emissions from phase_one()")
print("first:", emissions[0].pattern_name, "site", emissions[0].site)
emissions[0]  # Emission: SOM + tagged product mols

44 emissions from phase_one()
first: add_carbonyl_o+single_to_double site 4


Emission("add_carbonyl_o+single_to_double", site=4)

## 5. BFS **`product_graph_bfs`** (bounded exploration)

**Warning:** default BFS can run a long time. Always set `max_nodes`, `max_depth`, and
prefer **`target=`** when exploring toward one product.

Default rules: **`product_graph_ruleset()`** (QuinoneFormation + EpoxideHydration + Phase I core,
**no** Tautomerization).

In [7]:
EUGENOL = "COc1ccc(O)cc1"
QUINONE_LIKE = "O=C1C=C(O)C(=O)C(O)=C1"

net = product_graph_bfs(
    EUGENOL,
    target=QUINONE_LIKE,
    max_nodes=64,
    max_depth=4,
)
print(
    "nodes:",
    net.n_nodes(),
    "edges:",
    net.n_edges(),
    "reaches target:",
    net.reaches(QUINONE_LIKE),
)
net  # MetabolicNetwork summary + start/target drawings

nodes: 48 edges: 130 reaches target: True


MetabolicNetwork(nodes=48, edges=130)

## 6. One **`MetabolicNetwork`** for BFS + search

Pass the same `network=` handle to **`product_graph_bfs`** and **`find_path`** so BFS layers
and search hops accumulate on one graph. Then **`step_plan_between`** summarizes routes
between node indices.

In [8]:
net = MetabolicNetwork()
product_graph_bfs("CC", target="CCO", max_nodes=32, max_depth=3, network=net)
hits, _, net = find_path("CC", "CCO", max_paths=1, max_nodes=200, network=net)
root = net.root_idx()
target_i = net.index_of(hits[0].smiles)
plan = net.step_plan_between(root, target_i)
print("ethanol index:", target_i, "plan steps:", len(plan))
print("parents of ethanol:", net.parents(hits[0].smiles))
plan  # StepPlan between root and target

ethanol index: 1 plan steps: 1
parents of ethanol: ['CC', 'CC']


StepPlan(1 steps, ~1 lin)

## 6b. **`GraphNode` / `GraphEdge`** views

Index the live graph without marshalling: ``net[i]`` is a node view; ``inbound_edge(slot)``
exposes rule, parent index, and tagged ``kept_mol``.

In [9]:
from xenosite.forest import GraphEdge, GraphNode  # noqa: F401 — re-exported pyclasses

eth = net[target_i]
edge = eth.inbound_edge(0)
print(edge.rule, "parent:", edge.parent_index, "→ child:", edge.child_index)
edge  # GraphEdge: SOM site on tagged kept_mol

Hydroxylation parent: 0 → child: 1


GraphEdge("Hydroxylation"@0, 0→1)

## 7. Tautomer normalization (chematic pick)

Enol/keto pairs can be aligned before search with `normalize_tautomer=True` on `find_path`.
Here we normalize reactants directly.

In [10]:
keto, _ = normalize_tautomer("CC=O")
enol, changed = normalize_tautomer("OC=C")
print("enol changed:", changed, "same CSMI:", enol.csmi == keto.csmi)
enol  # ForestMol after tautomer pick

enol changed: True same CSMI: True


ForestMol("CC=O")

## 8. Seeded **`random_path`** (pathway sampling)

Explores one stochastic walk under Phase I rules — useful for priors and fuzz tests.

In [20]:
walk = random_path(APAP, seed=420, max_steps=5)
print("n hops:", len(walk.path) - 1)
walk  # RandomPathOutcome

n hops: 5


RandomPathOutcome("C=1C(O)C(O)(C2OC2C=1O)N")

## 9. `ForestMol` notebook display

Public `ForestMol` carries **atom tags** (survivors vs born). Leave the mol as the
last expression so IPython renders `_repr_html_` (RDKit SVG when available; text
`__str__` always). `to_rdkit()` is a CSMI-only view — tags are not round-tripped.

In [12]:
# Prefer ForestMol notebook HTML (tags + SVG). RDKit Draw is optional.
mol = ForestMol(APAP)
print("csmi:", mol.csmi, "trace:", mol.trace_counts())
mol

csmi: c1cc(ccc1NC(C)=O)O trace: (12, 11, 0, 0)


ForestMol("c1cc(ccc1NC(C)=O)O")

## 10. Rulesets at a glance

| Factory | Use |
|---------|-----|
| `default_ruleset()` | `find_path` default (includes Tautomerization) |
| `phase_one()` | Flat Phase I catalog (QF + EH + leaves) |
| `product_graph_ruleset()` | BFS default (QF + EH + Phase I core, no tautomer) |

In [13]:
rs = phase_one()
print(rs.name, "members:", len(rs), "patterns:", len(rs.patterns()))
# Also try: default_ruleset() / product_graph_ruleset()
rs

PhaseOne members: 18 patterns: 77


RuleSet("PhaseOne", 18 members, 77 patterns)